## Data Generation Process

### Synthetic Field Construction

Synthetic property fields were constructed in Gaussian space by combining two independent components: a deterministic large-scale trend and a stochastic short-scale residual. The trend component was generated from 4 trend families selected to represent different forms of large-scale spatial structure, including smooth undulating surfaces, asymmetric ridge features, and broad stationary patterns. Each trend surface was generated on a 100 × 100 grid over a 1000 m × 1000 m domain, with a 10 m cell size. The trend was normalized to zero mean and scaled to control its contribution to the total field variance.

Residual fields were generated as Gaussian random fields using FFT-based spectral simulation. The residual component honored the specified variogram model, correlation range, partial sill, and nugget effect. The synthetic property field was then formed as the additive Gaussian-space combination of the trend and residual:

Z = Trend + Residual

The resulting field served as the reference property field used in the subsequent sampling and modeling workflow.

---

### Sampling and Experimental Design

Well locations were generated using a two-stage sampling scheme that combines broad spatial coverage with preferential sampling. In the first stage, wells were placed on a coarse regular grid to ensure coverage across the domain. In the second stage, additional candidate locations were drawn uniformly at random and accepted with a probability that increased with porosity. This mimics informative sampling in petroleum exploration, where wells are more likely to be drilled in promising reservoir locations.

Two sampling configurations were tested to evaluate the effect of preferential sampling on trend–residual dependence:

* 100% uniform sampling
* 25% uniform / 75% preferential sampling

Each configuration produced 324 wells. The experimental design crossed four trend surfaces, three residual correlation range ratios, three trend-to-total variance ratios, one variogram model, three nugget levels, and two sampling designs. This produced a suite of synthetic datasets spanning different spatial structures, signal-to-noise conditions, measurement error levels, and sampling biases relevant to subsurface characterization.


---

### Factorial Parameters

| Parameter | Symbol / Label | Values | Purpose |
|---|---|---|---|
| **Trend families** | — | Channel, Ridge, GRF   | 4 diverse trend structures |
| **Range ratio** | rr = range / L | 0.10, 0.2, 0.30 | Residual spatial continuity relative to domain |
| **Variance ratio** | vr = Var(trend) / Var(Z) | 0.4, 0.5, 0.6, 0.7 | Signal strength: trend-dominated to residual-dominated |
| **Variogram model** | — | Exponential, Spherical | Residual covariance structure |
| **Sampling design** | f_regular | 1.00 (100% uniform), 0.5 (50/50) , 0.25 (25/75) | Preferential sampling intensity |
| **Sample size** | N | 324 wells | Fixed per dataset |
| **Preferential threshold** | P_threshold | 0.8 | Acceptance probability scaling for preferential (biased) sampling |

In [ ]:
import json
import os

# Core numerical and tabular tools
import math                              # Basic mathematical functions
import itertools                         # Cartesian products for experimental design
from itertools import product as iterproduct
import numpy as np                       # Array operations and gridded data
import pandas as pd                      # Tabular data handling

# Progress-bar control
from tqdm import tqdm                    # Progress bars
from functools import partialmethod      # Used to modify tqdm defaults

# Warning control
ignore_warnings = True                   # Set to False to show warnings

if ignore_warnings:
    import warnings
    warnings.filterwarnings("ignore")

# Plotting and visualization
import matplotlib as mpl
import matplotlib.pyplot as plt          # Main plotting library
import seaborn as sns                    # Statistical plotting
from matplotlib.colors import Normalize  # Normalize values for colormaps
from matplotlib.ticker import (
    MultipleLocator,
    AutoMinorLocator
)                                        # Control major and minor axis ticks
from pathlib import Path
plt.rc("axes", axisbelow=True)           # Draw grid lines below plot elements
cmap = plt.cm.viridis                    # Default colormap

# Spatial filtering and convolution
import scipy.signal as signal            # Moving-window kernels and signal operations
from scipy.ndimage import gaussian_filter # Gaussian smoothing
from astropy.convolution import convolve # General convolution operations

# Geostatistical tools
import geostatspy                        # Geostatistical workflows
import geostatspy.GSLIB as GSLIB         # GSLIB utilities and visualization wrappers
import geostatspy.geostats as geostats   # Geostatistical methods translated from GSLIB


### **Geostats Related Helper Functions**

In [ ]:
def scov(A, B):
    """Sample spatial covariance between two 2D arrays."""
    a = (A - A.mean()).ravel()
    b = (B - B.mean()).ravel()
    return float((a * b).mean())

def var1(x):
    x = np.asarray(x)
    return float(np.var(x, ddof=1))

def cov1(x,y):
    x = np.asarray(x) - np.mean(x)
    y = np.asarray(y) - np.mean(y)
    return float(np.dot(x,y) / (len(x)-1))
    
# variogram models
def variogram_model(h, model, range_, sill, nugget):
    """Create an analytical variogram model given its parameters."""
    c = sill - nugget
    hr = np.asarray(h, float) / range_
    if model.lower().startswith("exp"):
        gamma = nugget + c * (1.0 - np.exp(-3*hr))
    elif model.lower().startswith("gau"):
        gamma = nugget + c * (1.0 - np.exp(-3*(hr ** 2)))
    elif model.lower().startswith("sph"):
        gamma = np.where(
            hr <= 1.0,
            nugget + c * (1.5 * hr - 0.5 * hr**3),
            nugget + c,
        )
    else:
        raise ValueError("Model must be exponential, gaussian, or spherical")
    return gamma

#calculating covariance from variogram
def covariance_from_variogram(h, model, range_, sill, nugget):
    """Calculate covariance from specific analytical variogram model."""
    gamma = variogram_model(h, model, range_, sill, nugget)
    return (sill - nugget) - (gamma - nugget)

In [ ]:
# =============================================================================
# Core Function: Gram-Schmidt
# =============================================================================
# ====== Gram-Schmidt Helpers ======
def demean(A):
    """Return A with spatial mean removed."""
    return A - float(np.mean(A))

def rescale_to_var(A, v_target, eps=1e-12):
    """Center and scale A to have sample variance v_target."""
    A0 = A - A.mean()
    v  = float(A0.var())
    if v < eps:
        return A0  # nothing to scale (or degenerate); returns zero-mean A
    return A0 * np.sqrt(max(v_target, 0.0) / v)

def norm2(v, eps=1e-12):
    """Euclidean norm with floor to avoid division by zero."""
    return float(np.sqrt(np.dot(v, v))) + eps
    
def gram_schmidt_orthogonalization(A, B, eps=1e-12):
    """
    Make A (residual) orthogonal to B (trend) via Gram-Schmidt,
    keeping B fixed as the reference direction.

    Concept:
        Let a = demean(A), b = demean(B) (flattened).
        Remove the projection of a onto b:
            a_perp = a - alpha * b
        where:
            alpha = <a, b> / <b, b>

    This guarantees:
        <a_perp, b> = 0   (up to numerical precision)

    Note:
        A_orth + B ≠ A + B in general (Z is not exactly preserved —
        this is mathematically inherent to the GS correction).
        The correction term is alpha * B0.

    Args:
        A : ndarray  — residual field (to be orthogonalized)
        B : ndarray  — trend field    (kept fixed as reference)

    Returns:
        A_orth : A with trend-projection removed, mean restored
        B_orth : B unchanged (copy)
    """
    if A.shape != B.shape:
        raise ValueError(f"A and B must have the same shape, got {A.shape} vs {B.shape}")
    if A.ndim not in (1, 2):
        raise ValueError(f"Expected A, B to be 1D or 2D arrays; got ndim={A.ndim}")

    # Work in zero-mean space
    A0 = demean(A)
    B0 = demean(B)

    a = A0.ravel()
    b = B0.ravel()

    bb = float(np.dot(b, b))

    if bb < eps:
        # Degenerate reference: trend has zero variance, nothing to orthogonalize against
        print("Degenerate reference: trend has zero variance, returning inputs unchanged.")
        return A.copy(), B.copy()

    alpha = float(np.dot(a, b)) / (bb + eps)
    a_perp = a - alpha * b


    # Restore original mean of residual; trend is unchanged
    A_orth = a_perp.reshape(A.shape) + A.mean()
    B_orth = B.copy()

    return A_orth, B_orth



### **Data Generation Helper Function**

In [ ]:
def torus_distance_grid(nx, ny, dx):
    """Compute periodic pairwise distances on a 2D grid.

    The distances wrap around the domain edges, creating a toroidal grid.
    This is used for FFT-based simulation, where periodicity is assumed.
    """
    ix = np.arange(nx)
    iy = np.arange(ny)

    # Shortest wrapped distance from the origin in each direction
    dx_idx = np.minimum(ix, nx - ix) * dx
    dy_idx = np.minimum(iy, ny - iy) * dx

    # Squared distance components for all grid locations
    X = dx_idx[:, None] ** 2
    Y = dy_idx[None, :] ** 2

    return np.sqrt(X + Y)


def simulate_grf_fft(nx, ny, dx, model, range_, sill, nugget, rng, pad_factor=2):
    """Simulate a Gaussian random field using FFT-based spectral simulation.

    A covariance model is converted to a power spectrum, random spectral
    coefficients are generated, and an inverse FFT produces the spatial field.
    Padding is used to reduce edge artifacts before cropping back to the target grid.
    """

    # Pad the grid to the next power of two for efficient FFT and fewer edge artifacts
    pnx = int(2 ** math.ceil(np.log2(nx * pad_factor)))
    pny = int(2 ** math.ceil(np.log2(ny * pad_factor)))

    # Compute wrapped distances on the padded grid
    H = torus_distance_grid(pnx, pny, dx)

    # Convert the variogram model to a covariance function on the distance grid
    C = covariance_from_variogram(H, model, range_, sill, nugget)

    # Convert covariance to a non-negative power spectrum
    S = np.fft.rfftn(C)
    S = np.maximum(S.real, 0.0)

    # Generate complex Gaussian white noise in the spectral domain
    A = rng.standard_normal(S.shape)
    B = rng.standard_normal(S.shape)
    Z = (A + 1j * B) * np.sqrt(S / 2.0)

    # Transform the random spectrum back to real space
    f = np.fft.irfftn(Z, s=(pnx, pny)).real

    # Crop the center of the padded field back to the requested grid size
    sx = (pnx - nx) // 2
    sy = (pny - ny) // 2
    field = f[sx : sx + nx, sy : sy + ny]

    # Add independent nugget noise, if specified
    if nugget > 0:
        field += rng.standard_normal((nx, ny)) * np.sqrt(nugget)

    # Standardize the field to zero mean and target variance
    field -= np.mean(field)
    field *= np.sqrt(sill / np.var(field))

    return field


def create_residual_component(
    seed,
    target_variance,
    variogram_type,
    range_,
    nugget,
    ny=100,
    nx=100,
    dx=10,
):
    """Create a synthetic residual field from a specified variogram model."""

    # Initialize the random number generator for reproducibility
    rng = np.random.default_rng(seed)

    # Use the target variance as the residual sill
    sill = target_variance
    model = variogram_type

    # Convert nugget from a fraction of the sill to variance units
    nugget_variance = nugget * sill

    # Simulate the residual Gaussian random field
    residual_field = simulate_grf_fft(
        nx,
        ny,
        dx,
        model,
        range_,
        sill,
        nugget_variance,
        rng,
    )

    return residual_field

In [ ]:
# =============================================================================
# Utility Functions
# =============================================================================

def make_Gaussian_kernel(sigma, nc, csiz):
    """Create a normalized 2D Gaussian kernel for spatial smoothing."""

    # Define kernel coordinates centered around zero
    x = np.arange(-nc // 2 + 1, nc // 2 + 1) * csiz
    xx, yy = np.meshgrid(x, x)

    # Compute Gaussian weights from distance to the kernel center
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))

    # Normalize so the kernel sums to one
    return kernel / np.sum(kernel)


def tps_fit(points, values, lambda_smooth=1e-6):
    """Fit a thin plate spline surface through scattered control points."""

    # Convert control-point coordinates and values to numeric arrays
    P = np.asarray(points, float)
    v = np.asarray(values, float)
    n = P.shape[0]

    def U(r2):
        """Thin plate spline radial basis function."""

        # Avoid log(0) by only evaluating nonzero distances
        out = np.zeros_like(r2)
        m = r2 > 0
        out[m] = r2[m] * np.log(r2[m])
        return out

    # Compute squared pairwise distances between control points
    d2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=2)

    # Build the TPS kernel matrix with small smoothing for numerical stability
    K = U(d2) + lambda_smooth * np.eye(n)

    # Add the affine polynomial terms required by TPS interpolation
    Pmat = np.column_stack([np.ones(n), P])

    # Assemble and solve the full TPS linear system
    A = np.block([[K, Pmat], [Pmat.T, np.zeros((3, 3))]])
    rhs = np.concatenate([v, np.zeros(3)])
    sol = np.linalg.solve(A, rhs)

    # Separate radial basis weights from affine coefficients
    w = sol[:n]
    a = sol[n:]

    def f(X, Y):
        """Evaluate the fitted TPS surface on a grid."""

        # Stack grid coordinates for vectorized distance calculations
        XY = np.stack([X, Y], axis=-1)
        g = np.zeros_like(X, dtype=float)

        # Sum radial basis contributions from each control point
        for i in range(n):
            dx = XY[..., 0] - P[i, 0]
            dy = XY[..., 1] - P[i, 1]
            r2 = dx * dx + dy * dy
            Ui = np.where(r2 > 0, r2 * np.log(r2), 0.0)
            g += w[i] * Ui

        # Combine affine plane and radial basis terms
        return a[0] + a[1] * X + a[2] * Y + g

    return f


def gaussian_blur_fft(field, dx, sigma):
    """Apply Gaussian smoothing using FFT-based convolution."""

    nx, ny = field.shape

    # Compute wrapped distance coordinates for the convolution kernel
    ix = np.arange(nx)
    iy = np.arange(ny)
    dx_idx = np.minimum(ix, nx - ix) * dx
    dy_idx = np.minimum(iy, ny - iy) * dx

    # Build the Gaussian kernel on the periodic grid
    H2 = dx_idx[:, None]**2 + dy_idx[None, :]**2
    K = np.exp(-0.5 * H2 / (sigma * sigma))

    # Convolve the field with the Gaussian kernel in Fourier space
    F = np.fft.rfftn(field)
    Khat = np.fft.rfftn(K)
    out = np.fft.irfftn(F * Khat, s=(nx, ny)).real

    return out


# =============================================================================
# GRF-Based Trend Generator
# =============================================================================

def generate_grf_trends(create_residual_component, configs):
    """Generate smooth Gaussian-random-field-based trend surfaces."""

    trends = {}

    for cfg in configs:
        # Copy the configuration so the original input list is not modified
        cfg = cfg.copy()

        # Use the provided label, or create one from the seed
        label = cfg.pop("label", f"GRF-{cfg['seed']}")

        # Generate and store the trend surface
        trends[label] = create_residual_component(**cfg)

    return trends

In [ ]:
def generate_tps_trend(
    L,
    dx,
    anchors_xy,
    anchors_val,
    lambda_smooth=1e-6,
    R=600.0,
    target_variance=0.7,
):
    """Generate a smooth trend surface using TPS interpolation and Gaussian blurring."""

    # Define the grid dimensions from the domain size and cell spacing
    nx = int(L / dx)
    ny = int(L / dx)

    # Create grid coordinates over the square domain
    x = np.linspace(0, L, nx)
    y = np.linspace(0, L, ny)
    X, Y = np.meshgrid(x, y, indexing="ij")

    # Fit a thin plate spline surface through the anchor points
    f = tps_fit(
        np.asarray(anchors_xy),
        np.asarray(anchors_val),
        lambda_smooth,
    )

    # Evaluate the fitted TPS surface on the grid
    trend = f(X, Y)

    # Convert blur radius to Gaussian standard deviation
    sigma = R / (2 * np.sqrt(3.0))

    # Smooth the TPS surface to remove local interpolation artifacts
    trend = gaussian_blur_fft(trend, dx, sigma)

    # Center the trend to zero mean
    trend -= trend.mean()

    # Rescale the trend to the target variance
    v = trend.var()
    if v > 0:
        trend *= np.sqrt(target_variance / v)

    return trend, X, Y

In [ ]:
# =============================================================================
# Channel-Based Trend Generator
# =============================================================================

def generate_channel_trend(
    L,
    dx,
    R=200.0,
    target_variance=0.7,
    main_width=60.0,
    oxbow_width=45.0,
    meander_amplitude=350.0,
    meander_freq=2.0,
    oxbow_offset=0.6,
    angle=0.2,
):
    """Generate a smooth meandering-channel trend with an abandoned oxbow feature."""

    # Define the grid dimensions from the domain size and cell spacing
    nx = int(L / dx)
    ny = int(L / dx)

    # Create grid coordinates over the square domain
    x = np.linspace(0, L, nx)
    y = np.linspace(0, L, ny)
    X, Y = np.meshgrid(x, y, indexing="ij")

    # Rotate the coordinate system to orient the channel diagonally
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    Xr = cos_a * (X - 0.5 * L) + sin_a * (Y - 0.5 * L) + 0.5 * L
    Yr = -sin_a * (X - 0.5 * L) + cos_a * (Y - 0.5 * L) + 0.5 * L

    # Normalize the along-channel coordinate to the interval [0, 1]
    t = Xr / L

    # Define the original meandering channel centerline
    full_centerline = 0.5 * L + meander_amplitude * np.sin(
        meander_freq * np.pi * t
    )

    # Define the start and end of the neck-cutoff interval
    cutoff_start = oxbow_offset - 0.15
    cutoff_end = oxbow_offset + 0.15

    # Compute the channel-centerline elevations at both cutoff endpoints
    y_at_start = 0.5 * L + meander_amplitude * np.sin(
        meander_freq * np.pi * cutoff_start
    )
    y_at_end = 0.5 * L + meander_amplitude * np.sin(
        meander_freq * np.pi * cutoff_end
    )

    # Build a straight shortcut across the cutoff interval
    blend = np.clip((t - cutoff_start) / (cutoff_end - cutoff_start), 0, 1)
    straight_line = y_at_start + blend * (y_at_end - y_at_start)

    # Create a smooth mask that limits the shortcut to the cutoff interval
    mask_in = np.clip((t - (cutoff_start - 0.03)) / 0.03, 0, 1)
    mask_out = np.clip(((cutoff_end + 0.03) - t) / 0.03, 0, 1)
    cutoff_mask = mask_in * mask_out

    # Replace the original meander with the shortcut inside the cutoff zone
    active_centerline = (
        (1 - cutoff_mask) * full_centerline
        + cutoff_mask * straight_line
    )

    # Convert distance from the active channel into a smooth channel body
    dist_active = np.abs(Yr - active_centerline)
    channel_active = np.exp(-(dist_active**2) / (2 * main_width**2))

    # Preserve the abandoned meander loop as a weaker oxbow feature
    dist_oxbow = np.abs(Yr - full_centerline)
    channel_oxbow = np.exp(-(dist_oxbow**2) / (2 * oxbow_width**2))

    # Restrict and taper the oxbow so it fades near the cutoff endpoints
    oxbow_taper = np.sin(
        np.pi * np.clip((t - cutoff_start) / (cutoff_end - cutoff_start), 0, 1)
    )
    channel_oxbow *= cutoff_mask * oxbow_taper * 0.6

    # Add higher values along bends to mimic point-bar deposits
    curvature = np.abs(np.sin(meander_freq * np.pi * t))
    point_bar = 0.2 * curvature * channel_active

    # Combine active channel, abandoned oxbow, and point-bar contributions
    Z = channel_active + channel_oxbow + point_bar

    # Add low-amplitude floodplain variation so the background is not flat
    bg_freq = 1.5 * np.pi / L
    floodplain_base = (
        0.3
        + 0.1 * np.sin(bg_freq * Xr + 0.5)
        + 0.1 * np.cos(bg_freq * Yr * 1.3)
    )

    # Add levee-like enrichment that decays away from the channel system
    floodplain_dist = np.minimum(dist_active, dist_oxbow)
    levee = 0.25 * np.exp(-(floodplain_dist**2) / (2 * (2 * main_width) ** 2))

    # Add floodplain and levee components to the channel trend
    Z += floodplain_base + levee

    # Smooth the field to produce a continuous large-scale trend
    sigma_cells = R / dx
    Z = gaussian_filter(Z, sigma=sigma_cells)

    # Center the trend to zero mean
    Z -= Z.mean()

    # Rescale the trend to the target variance
    v = Z.var()
    if v > 0:
        Z *= np.sqrt(target_variance / v)

    return Z, X, Y

### Sampling and Dataset Creation Helper Functions

In [ ]:
def sample_wells_with_bias(
    nested,
    residual,
    trend,
    Lx,
    Ly,
    N_total=100,
    f_regular=0.2,
    P_threshold=0.20,
    seed=42,
    max_iterations=500000,
    batch_size=1000,
):
    """
    Generate well samples using regular coverage and optional preferential bias.
    
    Special case:
        f_regular == 1.5 -> pure random sampling without regular-grid placement
                            and without preferential bias.

    Args:
        nested: The nested 2D grid (ny, nx) — e.g. measured porosity in beta space.
        residual: The residual 2D grid (ny, nx). Gaussian space.
        trend: The trend 2D grid (ny, nx). Gaussian space.
        Lx: Domain size in x (m).
        Ly: Domain size in y (m).
        N_total: Total number of samples (wells) to generate.
        f_regular: Fraction of N_total to place on a regular grid (normally 0 to 1).
                   Use 1.5 to denote pure random sampling.
        P_threshold: Threshold value (0-1) for scaled data to promote acceptance
                     (scaled values >= P_threshold are always accepted).
        seed: Random seed for reproducibility.
        max_iterations: Safety limit on total proposed points to prevent infinite loops.
        batch_size: Number of candidate points proposed per iteration in rejection sampling.

    """

    # Initialize random number generator for reproducibility
    rng = np.random.default_rng(seed)

    # Get grid dimensions and total number of available cells
    ny, nx = nested.shape
    n_cells = nx * ny

    # Scale the sampled property to [0, 1] so the threshold is comparable across fields
    nested_min = nested.min()
    nested_max = nested.max()

    if nested_max - nested_min == 0:
        nested_scaled = np.ones_like(nested)
    else:
        nested_scaled = (nested - nested_min) / (nested_max - nested_min)

    # Create coordinate grids for the simulation domain
    x_grid = np.linspace(0.0, Lx, nx)
    y_grid = np.linspace(0.0, Ly, ny)
    coords_x_2d, coords_y_2d = np.meshgrid(x_grid, y_grid, indexing="xy")

    # Flatten grids so each cell can be sampled by a single index
    all_x = coords_x_2d.ravel()
    all_y = coords_y_2d.ravel()
    all_nested = nested.ravel()
    all_nested_scaled = nested_scaled.ravel()
    all_res = residual.ravel()
    all_trend = trend.ravel()

    # Storage for selected well locations and corresponding field values
    sampled_indices = set()
    x_samples, y_samples, n_samples = [], [], []
    r_samples, t_samples = [], []

    def _accept(idx):
        """Store one accepted sample location and its associated values."""

        sampled_indices.add(idx)
        x_samples.append(all_x[idx])
        y_samples.append(all_y[idx])
        n_samples.append(all_nested[idx])
        r_samples.append(all_res[idx])
        t_samples.append(all_trend[idx])

    # Special case: pure random sampling with no regular grid and no preferential bias
    if f_regular == 1.5:
        if N_total > n_cells:
            raise ValueError(
                f"N_total={N_total} exceeds number of grid cells ({n_cells})."
            )

        # Select unique grid cells uniformly at random
        chosen = rng.choice(n_cells, size=N_total, replace=False)

        # Store the selected samples
        for idx in chosen:
            _accept(int(idx))

        return (
            np.array(x_samples),
            np.array(y_samples),
            np.array(n_samples),
            np.array(r_samples),
            np.array(t_samples),
        )

    # Ensure the regular-sampling fraction is valid in normal sampling mode
    if not (0.0 <= f_regular <= 1.0):
        raise ValueError(
            "f_regular must be in [0, 1], or exactly 1.5 for pure random sampling."
        )

    # -------------------------------------------------------------------------
    # Phase 1: regular-grid sampling for broad spatial coverage
    # -------------------------------------------------------------------------

    # Determine the number of samples assigned to the regular grid
    N_regular = int(N_total * f_regular)

    # Approximate a square regular grid using the requested number of regular samples
    n_reg_side = int(np.sqrt(N_regular))

    # Select interior grid indices to avoid placing regular wells on domain boundaries
    reg_indices_y = np.round(
        np.linspace(0, ny - 1, max(3, n_reg_side + 2))
    ).astype(int)[1:-1]

    reg_indices_x = np.round(
        np.linspace(0, nx - 1, max(3, n_reg_side + 2))
    ).astype(int)[1:-1]

    # Remove duplicate indices that can occur after rounding
    reg_indices_y = np.unique(reg_indices_y)
    reg_indices_x = np.unique(reg_indices_x)

    # Add regular-grid samples until the regular-sampling target is reached
    for i_y, i_x in itertools.product(reg_indices_y, reg_indices_x):
        if len(x_samples) >= N_regular:
            break

        idx = i_y * nx + i_x

        if idx not in sampled_indices:
            _accept(idx)

    # -------------------------------------------------------------------------
    # Phase 2: preferential rejection sampling biased toward high property values
    # -------------------------------------------------------------------------

    # Candidate pool for random sampling
    all_indices = np.arange(n_cells)

    # Limit the batch size to the total number of grid cells
    effective_batch = min(batch_size, n_cells)

    # Track the number of proposed candidates to prevent infinite loops
    total_proposed = 0

    while len(x_samples) < N_total:
        total_proposed += effective_batch

        # Stop if rejection sampling cannot reach the target efficiently
        if total_proposed > max_iterations:
            raise RuntimeError(
                f"Rejection sampling exceeded {max_iterations} proposals with only "
                f"{len(x_samples)}/{N_total} samples collected. Consider raising "
                f"P_threshold or max_iterations."
            )

        # Propose unique candidate cells uniformly at random
        proposed = rng.choice(all_indices, size=effective_batch, replace=False)

        # Remove cells that have already been sampled
        mask_new = np.array([p not in sampled_indices for p in proposed], dtype=bool)
        proposed = proposed[mask_new]

        if len(proposed) == 0:
            continue

        # Assign higher acceptance probability to higher scaled property values
        probs = np.clip(all_nested_scaled[proposed] / P_threshold, 0.0, 1.0)

        # Accept or reject candidates using the preferential probabilities
        accept_mask = rng.random(len(proposed)) <= probs
        accepted = proposed[accept_mask]

        # Store accepted candidates until the total sampling target is reached
        for idx in accepted:
            if len(x_samples) >= N_total:
                break

            _accept(int(idx))

    # Return sampled coordinates and corresponding nested, residual, and trend values
    return (
        np.array(x_samples),
        np.array(y_samples),
        np.array(n_samples),
        np.array(r_samples),
        np.array(t_samples),
    )

### **Visualization Helper Functions**

In [ ]:
# =============================================================================
# Master Visualization: 3 x 4 grid of ALL trends
# =============================================================================

def plot_all_trends(
    all_trends,
    L,
    cmap="inferno",
    suptitle="Synthetic Trend Surfaces",
    clabel="Value",
    nrows=3,
    ncols=4,
    figsize=None,
    save_path=None,
    dpi=600,
    show=True,
):
    """Plot a family of trend surfaces in a publication-quality multi-panel layout."""
    n = len(all_trends)
    max_panels = nrows * ncols
    items = list(all_trends.items())[:max_panels]

    if figsize is None:
        figsize = (4.6 * ncols, 4.0 * nrows)

    # Publication-oriented defaults local to this figure
    with mpl.rc_context({
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 14,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.03,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }):
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize, constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()

        # Shared color scale for direct comparison across panels
        vmin = min(np.nanmin(grid) for _, grid in items)
        vmax = max(np.nanmax(grid) for _, grid in items)

        for i, (label, grid) in enumerate(items):
            ax = axes[i]
            ny_g, nx_g = grid.shape

            x_coords = np.linspace(0, L, nx_g)
            y_coords = np.linspace(0, L, ny_g)
            Xp, Yp = np.meshgrid(x_coords, y_coords)

            im = ax.pcolormesh(
                Xp, Yp, grid,
                shading="auto",
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                rasterized=True,   # helpful for complex meshes in vector export
            )

            ax.set_title(label, fontweight="bold")
            ax.set_xlabel("Easting (m)")
            ax.set_ylabel("Northing (m)")
            ax.set_aspect("equal")
            ax.set_xlim(0, L)
            ax.set_ylim(0, L)

            cbar = fig.colorbar(im, ax=ax, shrink=0.86, pad=0.02)
            cbar.set_label(clabel)

        for j in range(len(items), max_panels):
            axes[j].set_visible(False)

        fig.suptitle(suptitle, fontweight="bold")

        if save_path is not None:
            save_path = Path(save_path)
            save_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=dpi, facecolor="white")
            print(f"Saved figure to: {save_path.resolve()}")

        if show:
            plt.show()
        else:
            plt.close(fig)


def plot_grid(df, component_array, title):
    """Plot 2D property maps"""
    plt.figure(figsize=(6, 5))
    # overlay points (same cmap + norm so colors line up with the colorbar)
    pormin = df.Porosity.min(); pormax = df.Porosity.max(); 
    norm = Normalize(vmin=pormin, vmax=pormax)
    plt.scatter(df["X"], df["Y"], c=df["Porosity"], cmap="viridis",
    norm=norm, s=35, edgecolors="black", linewidths=0.5, alpha=0.95, zorder=3)
    im = plt.imshow(component_array, origin="lower", extent=[0, L, 0, L], aspect="equal",cmap="viridis")
    plt.xlabel("Easting (m)")
    plt.ylabel("Northing (m)")
    plt.title(title)
    plt.colorbar(im, label="Porosity")
    plt.show()



def add_grid():
    plt.gca().grid(True, which='major',linewidth = 1.0); plt.gca().grid(True, which='minor',linewidth = 0.2) # add y grids
    plt.gca().tick_params(which='major',length=7); plt.gca().tick_params(which='minor', length=4)
    plt.gca().xaxis.set_minor_locator(AutoMinorLocator()); plt.gca().yaxis.set_minor_locator(AutoMinorLocator()) # turn on minor ticks 

In [ ]:
def plot_samples(sampled_df, Z_beta, trend_beta, base_dir, dataset_name):
    Lx, Ly = 1000, 1000
    nx, ny = trend_beta.shape[1], trend_beta.shape[0]

    # Shared range for Z panel and sampled data panel
    z_vmin = min(np.nanmin(Z_beta), sampled_df["Porosity"].min())
    z_vmax = max(np.nanmax(Z_beta), sampled_df["Porosity"].max())

    # Independent range for trend panel
    trend_vmin = np.nanmin(trend_beta)
    trend_vmax = np.nanmax(trend_beta)

    plt.rcParams.update({
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "figure.titlesize": 16,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })

    fig, axes = plt.subplots(
        1, 3,
        figsize=(17, 5.8),
        constrained_layout=True
    )

    cmap = "viridis"
    extent = [0, Lx, 0, Ly]

    # Map sample coordinates to grid indices for trend lookup
    # Assumes sampled_df X,Y are in [0, 1000] and fields are on a regular grid over that domain.
    x_idx = np.clip(((sampled_df["X"].to_numpy() / Lx) * (nx - 1)).round().astype(int), 0, nx - 1)
    y_idx = np.clip(((sampled_df["Y"].to_numpy() / Ly) * (ny - 1)).round().astype(int), 0, ny - 1)

    trend_at_samples = trend_beta[y_idx, x_idx]

    trend_scatter_kwargs = dict(
        c=trend_at_samples,
        s=36,
        cmap=cmap,
        edgecolors="black",
        linewidths=0.5,
        alpha=0.95,
        zorder=10,
        vmin=trend_vmin,
        vmax=trend_vmax,
    )

    z_scatter_kwargs = dict(
        c=sampled_df["Porosity"],
        s=36,
        cmap=cmap,
        edgecolors="black",
        linewidths=0.5,
        alpha=0.95,
        zorder=10,
        vmin=z_vmin,
        vmax=z_vmax,
    )

    # --- Panel 1: Trend ---
    ax0 = axes[0]
    im0 = ax0.imshow(
        trend_beta,
        extent=extent,
        origin="lower",
        cmap=cmap,
        aspect="equal",
        vmin=trend_vmin,
        vmax=trend_vmax,
        interpolation="none"
    )
    ax0.scatter(sampled_df["X"], sampled_df["Y"], **trend_scatter_kwargs)
    ax0.set_title("Trend", fontsize=13, fontweight="bold")
    ax0.set_xlabel("Easting (m)")
    ax0.set_ylabel("Northing (m)")

    # --- Panel 2: Regionalized Variable ---
    ax1 = axes[1]
    im1 = ax1.imshow(
        Z_beta,
        extent=extent,
        origin="lower",
        cmap=cmap,
        aspect="equal",
        vmin=z_vmin,
        vmax=z_vmax,
        interpolation="none"
    )
    ax1.scatter(sampled_df["X"], sampled_df["Y"], **z_scatter_kwargs)
    ax1.set_title("Regionalized Variable (Z)", fontsize=13, fontweight="bold")
    ax1.set_xlabel("Easting (m)")
    ax1.set_ylabel("Northing (m)")

    # --- Panel 3: Sampled Data ---
    ax2 = axes[2]
    sc = ax2.scatter(
        sampled_df["X"],
        sampled_df["Y"],
        **z_scatter_kwargs
    )
    ax2.set_title("Sampled Data", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Easting (m)")
    ax2.set_ylabel("Northing (m)")
    ax2.set_xlim(0, Lx)
    ax2.set_ylim(0, Ly)
    ax2.set_aspect("equal")

    # Consistent axis formatting
    for ax in axes:
        ax.set_xlim(0, Lx)
        ax.set_ylim(0, Ly)
        ax.xaxis.set_major_locator(MultipleLocator(250))
        ax.yaxis.set_major_locator(MultipleLocator(250))
        ax.xaxis.set_minor_locator(MultipleLocator(125))
        ax.yaxis.set_minor_locator(MultipleLocator(125))
        ax.tick_params(axis="both", which="major", direction="out", length=5, width=0.8)
        ax.tick_params(axis="both", which="minor", direction="out", length=3, width=0.6)
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

    # Colorbar for Trend only
    cbar0 = fig.colorbar(
        im0,
        ax=ax0,
        location="right",
        shrink=0.92,
        pad=0.02
    )
    cbar0.set_label("Value", rotation=90)

    # Shared colorbar for Z and Sampled Data
    cbar1 = fig.colorbar(
        im1,
        ax=[ax1, ax2],
        location="right",
        shrink=0.92,
        pad=0.02
    )
    cbar1.set_label("Value", rotation=90)

    output_path = os.path.join(base_dir, dataset_name, "reconstructed_fields.png")
    plt.savefig(output_path, dpi=600, bbox_inches="tight")
    plt.close(fig)

### Generate Exhaustive Truth Fields

In [ ]:
# --- Grid Parameters ---
rng = np.random.default_rng(44)
L = 1000.0           # domain size in meters (both x and y)
dx = 10             # grid spacing in meters

# --- Grid ---
nx = int(L / dx)
ny = int(L / dx)

x_cordinates = np.arange(0.0, L, dx)
y_cordinates = np.arange(0.0, L, dx)


In [ ]:
# Asymmetric ridge: steep west face, gentle east slope
ridge_xy = np.array([
    [300, 500], [320, 300], [310, 700],   # ridge crest
    [100, 500], [100, 300], [100, 700],   # steep drop west
    [600, 500], [800, 400], [900, 500],   # gentle east
    [500, 100], [500, 900],               # ends
])
ridge_val = np.array([1.5, 1.3, 1.4,
                       -0.8, -0.6, -0.7,
                        0.5, 0.0, -0.3,
                        0.0, 0.0])

ridge_trend, _, _ = generate_tps_trend(L, dx=dx, anchors_xy=ridge_xy,
                                       anchors_val=ridge_val, R=600.0)


# --- 2. GRF-based trends ---
grf_configs = [
    dict(seed=3990,  target_variance=0.5, variogram_type="Gaussian",
         range_=650.0, nugget=0.0, label="GRF-3990"),
    dict(seed=6390, target_variance=0.5, variogram_type="Gaussian",
         range_=650.0, nugget=0.0, label="GRF-6390"),
]
grf_trends = generate_grf_trends(create_residual_component, grf_configs)


# --- 4. Channel trends ---
channel_trend, _, _ = generate_channel_trend(L, dx,
                                     meander_amplitude=280, meander_freq=1.8,
                                     main_width=70, oxbow_width=55,
                                     oxbow_offset=0.6, angle=0.4, R=180)

# --- Combine all into one dictionary ---
all_trends = {}
all_trends['Asymmetric Ridge'] = ridge_trend.T
all_trends['Channel Trend'] = channel_trend.T
all_trends['Gaussian Random Field - 1'] = grf_trends['GRF-3990']
all_trends['Gaussian Random Field - 2'] = grf_trends['GRF-6390']


# --- Plot porosity trends ---
plot_all_trends(all_trends, L, cmap=plt.cm.viridis,
                suptitle='Trend Surfaces',
                clabel='Trend',nrows=1, ncols=4,
                save_path="synthetic_trend_family.png",
                dpi=1200)

### Create Dataset stash

In [ ]:
# =============================================================================
# Main Pipeline
# =============================================================================

def generate_all_combinations(all_trends, create_residual_component,
                               L=1000.0,
                               range_ratios=None,
                               var_ratios=None,
                               variogram_types=None,
                               f_regulars=None,       
                               nuggets=None,         
                               base_output_dir='guassian_experiment',
                               N_wells=324,
                               P_threshold=0.7,
                               sampling_seed=8460,
                               residual_base_seed=1000,
                               trend_labels=list(all_trends.keys())):
    """
    Generate all combinatorial datasets: trend + residual.

    For each combination of (trend, range_ratio, var_ratio, variogram_type):
        1. Create output folder
        2. Generate residual field
        4. Combine: Z = trend + residual
        5. Sample wells with bias
        6. Save sampled DataFrame, grids (.npy), and metadata (.json)

    Returns
    -------
    summary : pd.DataFrame
        One row per combination with parameters and paths.
    """

    os.makedirs(base_output_dir, exist_ok=True)

    
    combos = list(iterproduct(trend_labels, range_ratios, var_ratios, variogram_types, f_regulars, nuggets))

    print(f"Total combinations: {len(combos)}")

    summary_rows = []
    pbar = tqdm(combos, desc="Generating datasets", unit="combo",
                bar_format="{l_bar}{bar:30}{r_bar}")

    for i, (trend_label, rr, vr, vario, f_regular, nugget) in enumerate(combos):

        # Update progress bar description
        pbar.set_postfix_str(f"{trend_label} | rr={rr} vr={vr}  {vario}")

        # --- Folder name ---
        safe_label = trend_label.replace(' ', '_').replace('-', '_')
        folder_name = f"{i:04d}_{safe_label}_rr{rr}_vr{vr}_fr{f_regular}_{vario}"
        folder_path = os.path.join(base_output_dir, folder_name)
        os.makedirs(folder_path, exist_ok=True)

        # --- Get trend ---
        porosity_trend = all_trends[trend_label]


        # --- Generate residual ---
        residual_range = rr * L
        residual_seed = residual_base_seed + i

        trend_var = np.var(porosity_trend)
        resid_var = trend_var * ((1 - vr) / vr)

        residual_field = create_residual_component(
            residual_seed, resid_var, vario, residual_range, nugget
        )

        R = residual_field
        T = porosity_trend

        # Orthogonalize trend against residual (residual as reference)
        T, R = gram_schmidt_orthogonalization(porosity_trend, residual_field, eps=1e-12)

        # Rescale to desired residual variance
        R *= np.sqrt(resid_var / np.var(R))

        Z_gaussian = T + R
        
        # --- Sample wells ---
        x_s, y_s, n_s, r_s, t_s = sample_wells_with_bias(
            nested=Z_gaussian,
            residual=R,
            trend=T,
            Lx=L, Ly=L,
            N_total=N_wells,
            f_regular=f_regular,
            P_threshold=P_threshold,
            seed=sampling_seed,
        )


        sampled_df = pd.DataFrame({
            'X': x_s, 'Y': y_s,
            'Porosity': n_s,
            'Res_Por': r_s,
            'Trend_Por': t_s,
        })

        # --- Save files ---
        sampled_df.to_csv(os.path.join(folder_path, 'sampled_wells.csv'), index=False)
        np.save(os.path.join(folder_path, 'trend_gaussian.npy'), T)
        np.save(os.path.join(folder_path, 'residual_gaussian.npy'), R)
        np.save(os.path.join(folder_path, 'Z_gaussian.npy'), Z_gaussian)

        plot_samples(sampled_df, Z_gaussian, T, base_output_dir, folder_name)

        ratio = np.var(porosity_trend) / np.var(Z_gaussian)
        # --- Metadata JSON ---
        metadata = {
            'combo_index': i,
            'trend_label': trend_label,
            'range_ratio': rr,
            'variance_ratio': vr,
            'variogram_type': vario,
            'residual_seed': residual_seed,
            'residual_range_m': residual_range,
            'folder': folder_path,
            'actual_trend_to_total_var': ratio,
            'f_regular': f_regular,
            'nugget': nugget,
        }

        with open(os.path.join(folder_path, 'metadata.json'), 'w') as f:
            json.dump(metadata, f, indent=2)

        # --- Summary row ---
        summary_rows.append(metadata)

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(base_output_dir, 'experiment_summary.csv'), index=False)
    print(f"\nDone. {len(combos)} datasets saved to '{base_output_dir}/'")
    print(f"Summary saved to '{base_output_dir}/experiment_summary.csv'")

    return summary_df

In [ ]:
base_dir = 'GS_experiment_datasets'
trends = list(all_trends.keys())
summary_df = generate_all_combinations(all_trends, create_residual_component,
                               L=1000.0,
                               range_ratios=[0.1, 0.15, 0.2],
                               var_ratios=[0.4, 0.5, 0.6, 0.7],
                               variogram_types=["Spherical"],  
                               f_regulars=[0.25, 0.5, 1],       # sampling configurations
                               nuggets=[0],          # nugget values for residual
                               base_output_dir=base_dir,
                               N_wells=324,
                               P_threshold=0.8,
                               sampling_seed=54653,
                               residual_base_seed=1300,
                               trend_labels=trends)

In [ ]:
summary_df = pd.read_csv(f"{base_dir}/experiment_summary.csv")
summary_df.head(10)